In [ ]:
%pip install --upgrade pip
%pip install langgraph langchain_openai
%pip install datasets
%pip install langchain-community rank_bm25
%pip install huggingface_hub
%pip install llama-index-tools-duckduckgo
%pip install ddgs


### Creating a RAG Tool for Guest Stories


In [7]:
import datasets
from langchain_core.documents import Document

# Load the dataset
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

# Convert dataset entries into Document objects
docs = [
    Document(
        page_content="\n".join([
            f"Name: {guest['name']}",
            f"Relation: {guest['relation']}",
            f"Description: {guest['description']}",
            f"Email: {guest['email']}"
        ]),
        metadata={"name": guest["name"]}
    )
    for guest in guest_dataset
]


In [8]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.tools import Tool

bm25_retriever = BM25Retriever.from_documents(docs)

def extract_text(query: str) -> str:
    """Retrieves detailed information about gala guests based on their name or relation."""
    results = bm25_retriever.invoke(query)
    if results:
        return "\n\n".join([doc.page_content for doc in results[:3]])
    else:
        return "No matching guest information found."

guest_info_tool = Tool(
    name="guest_info_retriever",
    func=extract_text,
    description="Retrieves detailed information about gala guests based on their name or relation."
)


In [12]:
import os

from typing import TypedDict, Annotated

from langgraph.graph.message import add_messages

from langchain_core.messages import AnyMessage, HumanMessage

from langgraph.prebuilt import ToolNode

from langgraph.graph import START, StateGraph

from langgraph.prebuilt import tools_condition

from langchain_openai import ChatOpenAI

from getpass import getpass



# Generate the chat interface, including the tools

openai_api_key = os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key: ")

chat = ChatOpenAI(

    model="gpt-4o-mini",

    api_key=openai_api_key,

)

tools = [guest_info_tool]

chat_with_tools = chat.bind_tools(tools)



# Generate the AgentState and Agent graph

class AgentState(TypedDict):

    messages: Annotated[list[AnyMessage], add_messages]



def assistant(state: AgentState):

    return {

        "messages": [chat_with_tools.invoke(state["messages"])],

    }



## The graph

builder = StateGraph(AgentState)



# Define nodes: these do the work

builder.add_node("assistant", assistant)

builder.add_node("tools", ToolNode(tools))



# Define edges: these determine how the control flow moves

builder.add_edge(START, "assistant")

builder.add_conditional_edges(

    "assistant",

    # If the latest message requires a tool, route to tools

    # Otherwise, provide a direct response

    tools_condition,

)

builder.add_edge("tools", "assistant")

alfred = builder.compile()



messages = [HumanMessage(content="Tell me about our guest named 'Lady Ada Lovelace'.")]

response = alfred.invoke({"messages": messages})



print("🎩 Alfred's Response:")

print(response["messages"][-1].content)


🎩 Alfred's Response:
Lady Ada Lovelace is your best friend and an esteemed mathematician known for her pioneering work in mathematics and computing. She is often celebrated as the first computer programmer due to her contributions to Charles Babbage's Analytical Engine. 

Here are her details:
- **Relation**: Best friend
- **Description**: Renowned for her work in mathematics and computing.
- **Email**: ada.lovelace@example.com

If you'd like to know more, just let me know!


### Building and Integrating Tools for Your Agent


In [15]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke("Who's the current President of France?")
print(results)


2 weeks ago - The president of France is the ex officio co-prince of Andorra, grand master of the Legion of Honour and of the National Order of Merit, and protector of the Institut de France in Paris. The officeholder is also honorary proto-canon of the Archbasilica of Saint John Lateran in Rome, although some have rejected the title in the past. The current ... 3 weeks ago - Macron later went to Puy du Fou ... the current government. On 30 August 2016, Macron resigned from the government ahead of the 2017 presidential election, to devote himself to his En Marche movement. There had been rising tensions and several reports that he had wanted to leave the Valls government since early 2015. He initially planned to leave after the cancellation of his "Macron 2" law but decided to stay on temporarily after a meeting with President François ... 2 days ago - A 1962 referendum held under the Fifth Republic at the request of President Charles de Gaulle transferred the election of the president

In [16]:
from langchain_core.tools import Tool
import random

def get_weather_info(location: str) -> str:
    """Fetches dummy weather information for a given location."""
    # Dummy weather data
    weather_conditions = [
        {"condition": "Rainy", "temp_c": 15},
        {"condition": "Clear", "temp_c": 25},
        {"condition": "Windy", "temp_c": 20}
    ]
    # Randomly select a weather condition
    data = random.choice(weather_conditions)
    return f"Weather in {location}: {data['condition']}, {data['temp_c']}°C"

# Initialize the tool
weather_info_tool = Tool(
    name="get_weather_info",
    func=get_weather_info,
    description="Fetches dummy weather information for a given location."
)


In [17]:
from langchain_core.tools import Tool
from huggingface_hub import list_models

def get_hub_stats(author: str) -> str:
    """Fetches the most downloaded model from a specific author on the Hugging Face Hub."""
    try:
        # List models from the specified author, sorted by downloads
        models = list(list_models(author=author, sort="downloads", direction=-1, limit=1))

        if models:
            model = models[0]
            return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
        else:
            return f"No models found for author {author}."
    except Exception as e:
        return f"Error fetching models for {author}: {str(e)}"

# Initialize the tool
hub_stats_tool = Tool(
    name="get_hub_stats",
    func=get_hub_stats,
    description="Fetches the most downloaded model from a specific author on the Hugging Face Hub."
)

# Example usage
print(hub_stats_tool.invoke("facebook")) # Example: Get the most downloaded model by Facebook


Error fetching models for facebook: HfApi.list_models() got an unexpected keyword argument 'direction'


### Integrating Tools with Alfred


In [18]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langchain_openai import ChatOpenAI

# Generate the chat interface, including the tools
chat = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
)
tools = [search_tool, weather_info_tool, hub_stats_tool]
chat_with_tools = chat.bind_tools(tools)

# Generate the AgentState and Agent graph
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {
        "messages": [chat_with_tools.invoke(state["messages"])],
    }

## The graph
builder = StateGraph(AgentState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message requires a tool, route to tools
    # Otherwise, provide a direct response
    tools_condition,
)
builder.add_edge("tools", "assistant")
alfred = builder.compile()

messages = [HumanMessage(content="Who is Facebook and what's their most popular model?")]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response["messages"][-1].content)


🎩 Alfred's Response:
### What is Facebook?

Facebook is an American social networking service owned by Meta Platforms. It was founded in 2004 by Mark Zuckerberg and his college roommates at Harvard University, including Eduardo Saverin, Andrew McCollum, Dustin Moskovitz, and Chris Hughes. The platform allows users to connect with friends and family, share updates, photos, and videos, join groups, and engage in various online activities. As of 2023, Facebook has over 2.8 billion monthly active users, making it one of the largest social platforms worldwide.

### Facebook's Most Popular Model

Unfortunately, I encountered an error when trying to retrieve information about Facebook's most popular model from the Hugging Face Hub. If you can provide specifics about the type of model you are interested in (e.g., AI models, datasets, etc.), I can try to find that information for you.


### The Complete Agent


In [19]:
import os
from getpass import getpass
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import START, StateGraph
from langchain_openai import ChatOpenAI

# Initialize the OpenAI model and combine all notebook tools
openai_api_key = os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key: ")
chat = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
)
tools = [guest_info_tool, search_tool, weather_info_tool, hub_stats_tool]
chat_with_tools = chat.bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {
        "messages": [chat_with_tools.invoke(state["messages"])],
    }

builder = StateGraph(AgentState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")
alfred = builder.compile()


In [20]:
### Example 1: Finding Guest Information

response = alfred.invoke({"messages": "Tell me about 'Lady Ada Lovelace'"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)


🎩 Alfred's Response:
Lady Ada Lovelace, born Augusta Ada Byron, is recognized as one of the first computer programmers in history. She was an English mathematician and a close associate of Charles Babbage, with whom she collaborated on the design of the Analytical Engine, a prototype for a digital computer. Her work in 1843 included writing a program for this machine, highlighting her advanced understanding of algorithms.

Born on December 10, 1815, Ada was the daughter of the famous poet Lord Byron and Anne Isabella Milbanke. Despite her father's literary fame, she had limited contact with him, as they separated shortly after her birth. 

In 1835, she married William King, who became the 8th Baron King, and later the Earl of Lovelace, making her Lady Lovelace. Together they had three children. Throughout her life, Lovelace faced various health issues, including asthma and stomach problems, and she struggled with the use of strong painkillers.

Ada Lovelace passed away on November 27, 

In [21]:
### Example 2: Checking the Weather for Fireworks

response = alfred.invoke({"messages": "What's the weather like in Paris tonight? Will it be suitable for our fireworks display?"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)


🎩 Alfred's Response:
Tonight in Paris, the weather is rainy with a temperature of 15°C. Unfortunately, this may not be suitable for your fireworks display.


In [22]:
### Example 3: Impressing AI Researchers

response = alfred.invoke({"messages": "One of our guests is from Qwen. What can you tell me about their most popular model?"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)


🎩 Alfred's Response:
It seems that I'm currently unable to retrieve information about the most popular model from Qwen due to a technical issue with the data source. However, Qwen is known for developing advanced language models. If you have specific questions or need information about Qwen's models in general, feel free to ask!


In [23]:
### Example 4: Combining Multiple Tools

response = alfred.invoke({"messages":"I need to speak with 'Dr. Nikola Tesla' about recent advancements in wireless energy. Can you help me prepare for this conversation?"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)


🎩 Alfred's Response:
Here are some recent advancements in wireless energy that you can discuss with Dr. Nikola Tesla:

1. **Electrodynamic Wireless Power Transmission**: Recent developments in this area focus on improving the efficiency of wireless power transfer using magnetic resonance and inductive coupling. These technologies are particularly notable for their applications in electric vehicle (EV) charging.

2. **Weather-Resistant Power Systems**: Breakthroughs are being made towards creating more robust wireless energy systems that can withstand various conditions, which could potentially power drones, assist in disaster relief operations, or even harness energy from space.

3. **RF Energy Harvesting**: The evolution of wireless communication technologies has enhanced the power density that can be transferred. This progress is enabling more efficient harvesting of radio frequency (RF) energy.

4. **Near-Field Wireless Power Transfer**: This emerging technology allows for energy tr

In [24]:
### Advanced Features: Conversation Memory

# First interaction
response = alfred.invoke({"messages": [HumanMessage(content="Tell me about 'Lady Ada Lovelace'. What's her background and how is she related to me?")]})


print("🎩 Alfred's Response:")
print(response['messages'][-1].content)
print()

# Second interaction (referencing the first)
response = alfred.invoke({"messages": response["messages"] + [HumanMessage(content="What projects is she currently working on?")]})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)


🎩 Alfred's Response:
Lady Ada Lovelace, renowned for her pioneering work in mathematics and computing, is your best friend. She is often celebrated as the first computer programmer due to her innovative work on Charles Babbage's Analytical Engine. If you'd like to reach out to her, you can contact her at ada.lovelace@example.com.

Would you like to know more about her contributions to mathematics and computing?

🎩 Alfred's Response:
Currently, Ada Lovelace is involved in several projects that focus on the intersection of technology and society, particularly in relation to data and AI. Some highlights include:

1. **Research Approaches to Data and AI**: She is part of projects that explore how data and AI can affect people's lives and institutions, aiming to strengthen the connections between research and the communities it supports.

2. **Ada Lovelace Day**: Celebrated on October 10, this event continues to promote the contributions of women in science, technology, engineering, and mat